# 🎬🍿 CGV 무비차트 크롤링 프로젝트

# 프로젝트 개요

CGV 무비차트 페이지의 공개 데이터를 동적으로 수집하고,
수집한 원천 데이터를 JSON으로 중간 저장한 후
Pandas를 이용하여 전처리하고 MySQL 데이터베이스에 저장한다.


# 요구사항 정의

## 기능 요구사항

| ID | 요구사항 | 내용 |
|---|---|---|
| FR-01 | 동적 페이지 접근 | Selenium으로 CGV 무비차트에 접속 |
| FR-02 | 영화 목록 수집 | 순위, 영화명, 관람 등급, 포스터 URL, 상세 URL 수집 |
| FR-04 | 중간 저장 | 크롤링 직후 원본 데이터를 Raw CSV로 저장 |
| FR-05 | 데이터 점검 | 행/열, 자료형, 결측치, 중복 확인 |
| FR-06 | 전처리 | 문자열 정리, 숫자형 변환, 날짜 표준화, 중복 제거 |
| FR-07 | CSV 저장 | 전처리 결과를 UTF-8-SIG CSV로 저장 |
| FR-08 | MySQL 저장 | `.env` 접속정보를 이용해 `movie_chart` DB의 영화 테이블에 저장 |
| FR-09 | 결과 확인 | CSV와 MySQL의 저장 건수 및 주요 컬럼 확인 |

## 수집 데이터

- 순위
- 영화명
- 에그지수
- 누적관객수
- 관람등급
- 포스터 URL
- 수집일시
- 출처 URL

## 저장 구조

- **중간 저장:** `data/raw/cgv_movie_raw.csv`
- **최종 파일 저장:** `data/cgv_movie_clean.csv`
- **DB 저장:** MySQL `movie_chart.cgv_movies`
- DB 접속정보는 `.env`의 `DB_HOST`, `DB_PORT`, `DB_USER`, `DB_PASSWORD`, `DB_NAME`을 사용한다.

# 동적 웹페이지 크롤링

https://cgv.co.kr/cnm/cgvChart/movieChart?tabParam=144

# 기본 설정

In [1]:
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin
from pymysql.cursors import DictCursor
import re
import time

import pandas as pd
import matplotlib.pyplot as plt

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException

plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

RAW_CSV_PATH = RAW_DIR / "cgv_movie_raw.csv"
FINAL_CSV_PATH = DATA_DIR / "cgv_movie_clean.csv"

TARGET_URL = "https://cgv.co.kr/cnm/cgvChart/movieChart?tabParam=144"

PAGE_LOAD_TIMEOUT = 30
WAIT_TIMEOUT = 20


# 크롬 웹브라우저 실행 후 종료

## 크롬 웹브라우저 실행

In [2]:
driver = webdriver.Chrome()

## URL 접속

In [3]:
driver.get(TARGET_URL)

## 크롬 웹브라우저 종료

In [4]:
driver.quit()
print('웹브라우저 종료')

웹브라우저 종료


# 드라이버 생성 함수 정의

In [5]:
def build_driver(headless=True):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1440,1200")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return driver


def clean_text(text):
    if text is None:
        return None
    text = re.sub(r"\\s+", " ", str(text)).strip()
    return text if text else None


def first_text(element, selectors):
    for selector in selectors:
        try:
            value = clean_text(element.find_element(By.CSS_SELECTOR, selector).text)
            if value:
                return value
        except Exception:
            pass
    return None


def normalize_url(url):
    if not url:
        return None
    return urljoin("https://cgv.co.kr", url)


In [6]:
def extract_age_rating(item):
    """차트 카드의 관람등급 이미지 alt만 추출한다."""
    try:
        imgs = item.find_elements(By.CSS_SELECTOR, "img[alt*='관람가']")
        for img in imgs:
            alt = clean_text(img.get_attribute("alt"))
            if alt:
                return alt

        # 관람등급 추출
        html = item.get_attribute("outerHTML") or ""
        match = re.search(
            r'alt=["\']([^"\']*(?:관람가|관람불가|청불)[^"\']*)["\']',
            html,
            flags=re.I
        )
        if match:
            return clean_text(match.group(1))

    except Exception as e:
        print("관람등급 추출 오류:", e)

    return None

def extract_release_info(item):
    """'재개봉' 또는 '개봉' 텍스트를 추출한다."""
    try:
        text = item.text
        match = re.search(r'(\d{4}\.\d{2}\.\d{2}\s*(?:개봉|재개봉))', text)
        if match:
            return clean_text(match.group(1))
    except Exception:
        pass
    return None


def collect_chart_list(driver):
    """CGV 무비차트에서 최종 산출물에 필요한 8개 항목을 수집한다."""
    driver.get(TARGET_URL)

    try:
        WebDriverWait(driver, WAIT_TIMEOUT).until(
            lambda d: len(d.find_elements(
                By.CSS_SELECTOR,
                "li[class*='bestChartList_chartItem__']"
            )) > 0
        )
    except TimeoutException:
        time.sleep(3)

    time.sleep(2)

    selectors = [
        "li[class*='bestChartList_chartItem__']",
        "li[class*='chartItem']",
    ]

    items = []
    for selector in selectors:
        items = driver.find_elements(By.CSS_SELECTOR, selector)
        if items:
            break

    print(f"차트 항목 수: {len(items)}")

    results = []
    crawled_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for rank, item in enumerate(items, start=1):
        try:
            title = None
            poster_url = None

            # 영화명과 포스터
            for img in item.find_elements(By.TAG_NAME, "img"):
                alt = clean_text(img.get_attribute("alt"))
                src = normalize_url(img.get_attribute("src"))

                if src and not poster_url:
                    poster_url = src

                if alt and "포스터" in alt:
                    title = alt.replace("포스터", "").strip()

            title_span = first_text(
                item,
                [
                    "span[class*='bestChartList_name__']",
                    "[class*='chartName']",
                ],
            )
            if title_span:
                title = title_span

            if not title:
                raise ValueError("영화명을 찾을 수 없음")

            info_text = clean_text(item.text) or ""

            # 에그지수
            egg_match = re.search(r"(\d+(?:\.\d+)?)\s*%", info_text)
            egg_index = egg_match.group(1) if egg_match else None

            # 누적관객수
            viewer_match = re.search(
                r"누적관객수\s*([\d,.]+\s*(?:만|천)?)",
                info_text
            )
            cumulative_viewers = (
                viewer_match.group(1) if viewer_match else None
            )

            # 관람등급
            age_rating = extract_age_rating(item)

            # 개봉/재개봉
            release_info_raw = extract_release_info(item)

            results.append({
                "rank": rank,
                "title": title,
                "egg_index_raw": egg_index,
                "cumulative_viewers_raw": cumulative_viewers,
                "release_info_raw": release_info_raw,
                "age_rating": age_rating,
                "poster_url": poster_url,
                "crawled_at": crawled_at,
                "source_url": TARGET_URL,
            })

            print(
                f"{rank}위 | {title} | "
                f"에그={egg_index or '-'} | "
                f"누적관객={cumulative_viewers or '-'} | "
                f"등급={age_rating or '-'}"
            )

        except Exception as e:
            print(f"{rank}위 수집 실패: {e}")

    return results


driver = build_driver(headless=True)

try:
    raw_records = collect_chart_list(driver)
finally:
    driver.quit()

raw_df = pd.DataFrame(raw_records)

print("\n" + "=" * 60)
print(f"차트 수집 완료: {len(raw_df)}건")
print("=" * 60)

display(raw_df.head(10))


차트 항목 수: 69
1위 | 오디세이 | 에그=98 | 누적관객=382.1만 | 등급=15세 관람가
2위 | 스파이더맨-브랜드 뉴 데이 | 에그=97 | 누적관객=696.5만 | 등급=12세 관람가
3위 | 사랑의 하츄핑-고래보석의 전설 | 에그=97 | 누적관객=57.5만 | 등급=전체 관람가
4위 | 명탐정 코난-하이웨이의 타천사 | 에그=93 | 누적관객=18.4만 | 등급=12세 관람가
5위 | 오케이 마담2 | 에그=77 | 누적관객=15.7만 | 등급=15세 관람가
6위 | 마루 밑 아리에티 | 에그=- | 누적관객=- | 등급=전체 관람가
7위 | 퍼피 구조대-더 다이노 무비 | 에그=99 | 누적관객=2.9만 | 등급=전체 관람가
8위 | 에이티즈 - 라이트 더 웨이 인 시네마 | 에그=- | 누적관객=- | 등급=전체 관람가
9위 | 호프 | 에그=85 | 누적관객=446만 | 등급=15세 관람가
10위 | 어떻게 해야 했을까 | 에그=97 | 누적관객=6.5만 | 등급=12세 관람가
11위 | BOYNEXTDOOR TOUR 'KNOCK ON Vol.2' IN JAPAN | 에그=- | 누적관객=- | 등급=전체 관람가
12위 | 터치드 콘서트 [하이라이트 포] - 더 무비 | 에그=- | 누적관객=- | 등급=전체 관람가
13위 | (라이브뷰잉)IDOLiSH7 VISIBLIVE TOUR '4WARD JOURNEY' Live Viewing | 에그=- | 누적관객=- | 등급=전체 관람가
14위 | 경주기행 | 에그=- | 누적관객=- | 등급=15세 관람가
15위 | 위커 맨-파이널 컷 | 에그=- | 누적관객=- | 등급=청소년 관람불가
16위 | 애정만세 | 에그=82 | 누적관객=5,946
 | 등급=15세 관람가
17위 | 인시디어스-그들이 넘어왔다 | 에그=- | 누적관객=- | 등급=15세 관람가
18위 | 몽상가들 | 에그=92 | 누적관객=4.5만 | 등급=청소년 관람불가
19위 | 백룸-익스텐디드 컷 | 에그=- | 누적관객

,rank,title,egg_index_raw,cumulative_viewers_raw,release_info_raw,age_rating,poster_url,crawled_at,source_url
0,1,오디세이,98,382.1만,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
1,2,스파이더맨-브랜드 뉴 데이,97,696.5만,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
2,3,사랑의 하츄핑-고래보석의 전설,97,57.5만,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
3,4,명탐정 코난-하이웨이의 타천사,93,18.4만,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
4,5,오케이 마담2,77,15.7만,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
5,6,마루 밑 아리에티,NaN,NaN,2026.08.19 재개봉,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
6,7,퍼피 구조대-더 다이노 무비,99,2.9만,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
7,8,에이티즈 - 라이트 더 웨이 인 시네마,NaN,NaN,2026.08.19 개봉,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
8,9,호프,85,446만,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
9,10,어떻게 해야 했을까,97,6.5만,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...


# 중간 저장 (Raw CSV)

수집 직후의 데이터를 **원본 그대로** `data/raw/cgv_movie_raw.csv`에 저장

In [7]:
# Raw CSV 중간 저장
RAW_DIR.mkdir(parents=True, exist_ok=True)

raw_preview_columns = [
    "rank", "title", "egg_index_raw", "cumulative_viewers_raw","release_info_raw",
    "age_rating", "poster_url", "crawled_at", "source_url"
]

raw_save_df = raw_df.copy()
for col in raw_preview_columns:
    if col not in raw_save_df.columns:
        raw_save_df[col] = pd.NA

raw_save_df = raw_save_df[raw_preview_columns].rename(columns={
    "egg_index_raw": "egg_index",
    "cumulative_viewers_raw": "cumulative_viewers_count",
})

raw_save_df.to_csv(RAW_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Raw CSV 저장 완료: {RAW_CSV_PATH}")
print(f"Raw CSV 컬럼: {raw_save_df.columns.tolist()}")
print(f"Raw CSV 건수: {len(raw_save_df)}건")


Raw CSV 저장 완료: D:\AI\data_analytics\crawling\cgv_crawling\data\raw\cgv_movie_raw.csv
Raw CSV 컬럼: ['rank', 'title', 'egg_index', 'cumulative_viewers_count', 'release_info_raw', 'age_rating', 'poster_url', 'crawled_at', 'source_url']
Raw CSV 건수: 69건


# 데이터 이해 (EDA)

전처리 전 원본 데이터의 구조와 품질을 확인

- 데이터 크기
- 컬럼 목록
- 자료형
- 결측치
- 중복 데이터

In [8]:
print("데이터 크기:", raw_df.shape)
print("\n컬럼 목록")
print(raw_df.columns.tolist())

print("\n자료형")
display(raw_df.dtypes)

print("\n결측치")
display(raw_df.isnull().sum().to_frame("missing_count"))

print("\n중복 행:", raw_df.duplicated().sum())

display(raw_df.head(10))


데이터 크기: (69, 9)

컬럼 목록
['rank', 'title', 'egg_index_raw', 'cumulative_viewers_raw', 'release_info_raw', 'age_rating', 'poster_url', 'crawled_at', 'source_url']

자료형


rank                      int64
title                       str
egg_index_raw               str
cumulative_viewers_raw      str
release_info_raw            str
age_rating                  str
poster_url                  str
crawled_at                  str
source_url                  str
dtype: object


결측치


,missing_count
rank,0
title,0
egg_index_raw,34
cumulative_viewers_raw,30
release_info_raw,39
age_rating,0
poster_url,0
crawled_at,0
source_url,0



중복 행: 0


,rank,title,egg_index_raw,cumulative_viewers_raw,release_info_raw,age_rating,poster_url,crawled_at,source_url
0,1,오디세이,98,382.1만,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
1,2,스파이더맨-브랜드 뉴 데이,97,696.5만,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
2,3,사랑의 하츄핑-고래보석의 전설,97,57.5만,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
3,4,명탐정 코난-하이웨이의 타천사,93,18.4만,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
4,5,오케이 마담2,77,15.7만,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
5,6,마루 밑 아리에티,NaN,NaN,2026.08.19 재개봉,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
6,7,퍼피 구조대-더 다이노 무비,99,2.9만,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
7,8,에이티즈 - 라이트 더 웨이 인 시네마,NaN,NaN,2026.08.19 개봉,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
8,9,호프,85,446만,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
9,10,어떻게 해야 했을까,97,6.5만,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...


# 데이터 전처리

## 전처리 항목

1. 영화명 공백 제거
2. 순위 숫자형 변환
3. 평점 숫자형 변환
4. 에그지수 숫자형 변환
5. 누적관객수의 `만/천` 단위를 숫자로 변환
6. 가격의 `원`, `,` 제거 후 숫자형 변환
7. 개봉일 날짜 형식 통일
8. 상세 URL 기준 중복 제거
9. 영화명이 없는 데이터 제거
10. 최종 컬럼 순서 정리

In [9]:
def parse_count(value):
    """317.9만, 1.1만, 123,456 등을 정수형 누적관객수로 변환."""
    if pd.isna(value):
        return pd.NA

    text = str(value).replace(",", "").strip()
    match = re.match(r"([\d.]+)\s*(만|천)?", text)

    if not match:
        return pd.NA

    number = float(match.group(1))
    unit = match.group(2)

    if unit == "만":
        number *= 10000
    elif unit == "천":
        number *= 1000

    return int(number)


def parse_number(value):
    if pd.isna(value):
        return pd.NA

    match = re.search(r"\d+(?:\.\d+)?", str(value).replace(",", ""))
    return float(match.group()) if match else pd.NA

def parse_release_info(text):
    """
    '2026.08.26 재개봉' -> ('재개봉', '2026-08-26')
    '2026.08.19 개봉'   -> ('개봉예정', '2026-08-19')
    NaN (이미 개봉됨)   -> ('개봉', pd.NA)
    """
    if pd.isna(text) or not str(text).strip():
        return "개봉", pd.NA  # 텍스트가 없는 경우: 이미 개봉 완료된 상태
    
    val = str(text)
    
    # 구분 값 설정: 재개봉 / 개봉예정
    if "재개봉" in val:
        release_type = "재개봉"
    elif "개봉" in val:
        release_type = "개봉예정"
    else:
        release_type = "개봉"
    
    # 날짜 추출 (YYYY.MM.DD)
    date_match = re.search(r'(\d{4})\.(\d{2})\.(\d{2})', val)
    if date_match:
        formatted_date = f"{date_match.group(1)}-{date_match.group(2)}-{date_match.group(3)}"
    else:
        formatted_date = pd.NA
        
    return release_type, formatted_date


def clean_age_rating(value):
    """HTML이 섞여 있어도 관람등급 텍스트만 남긴다."""
    if pd.isna(value):
        return pd.NA
    
    text = str(value)
    alt_match = re.search(r'alt=["\']([^"\']*(?:관람가|관람불가|청불)[^"\']*)["\']', text, flags=re.I)
    if alt_match:
        text = alt_match.group(1)
        
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", "", text).strip() 
    
    mapping = {
        "전체관람가": "전체 관람가",
        "12세관람가": "12세 관람가",
        "15세관람가": "15세 관람가",
        "18세관람가": "18세 관람가",
        "19세관람가": "19세 관람가",
        "청소년관람불가": "청소년 관람불가",
        "청불": "청소년 관람불가",
    }
    
    for key, value_str in mapping.items():
        if key in text:
            return value_str
            
    return text if text else pd.NA


def preprocess_movie_data(df):
    df = df.copy()

    df["title"] = df["title"].astype("string").str.strip()
    df = df[df["title"].notna() & (df["title"] != "")]

    df["rank"] = pd.to_numeric(
        df["rank"], errors="coerce"
    ).astype("Int64")

    
    df['egg_index'] = df['egg_index_raw'].apply(
        lambda x: f"{str(x).strip()}%" if pd.notna(x) and str(x).strip() not in ['', '-'] else None
    )
    
    df['cumulative_viewers_count'] = df['cumulative_viewers_raw'].apply(
        lambda x: str(x).strip() if pd.notna(x) and str(x).strip() not in ['', '-'] else None
    )

    df["age_rating"] = df["age_rating"].apply(clean_age_rating)

    df["poster_url"] = df["poster_url"].apply(normalize_url)

    df[["is_re_release", "release_date"]] = df["release_info_raw"].apply(
        lambda x: pd.Series(parse_release_info(x))
    )

    df = df.drop_duplicates(
        subset=["title"],
        keep="first"
    )

    final_columns = [
        "rank",
        "title",
        "egg_index",
        "cumulative_viewers_count",
        "is_re_release",
        "release_date",
        "age_rating",
        "poster_url",
        "crawled_at",
        "source_url",
    ]

    for col in final_columns:
        if col not in df.columns:
            df[col] = pd.NA

    return (
        df[final_columns]
        .sort_values("rank", na_position="last")
        .reset_index(drop=True)
    )


clean_df = preprocess_movie_data(raw_df)

print("전처리 후 데이터 크기:", clean_df.shape)
print("최종 컬럼:", clean_df.columns.tolist())

display(clean_df.head(10))


전처리 후 데이터 크기: (69, 10)
최종 컬럼: ['rank', 'title', 'egg_index', 'cumulative_viewers_count', 'is_re_release', 'release_date', 'age_rating', 'poster_url', 'crawled_at', 'source_url']


,rank,title,egg_index,cumulative_viewers_count,is_re_release,release_date,age_rating,poster_url,crawled_at,source_url
0,1,오디세이,98%,382.1만,개봉,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
1,2,스파이더맨-브랜드 뉴 데이,97%,696.5만,개봉,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
2,3,사랑의 하츄핑-고래보석의 전설,97%,57.5만,개봉,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
3,4,명탐정 코난-하이웨이의 타천사,93%,18.4만,개봉,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
4,5,오케이 마담2,77%,15.7만,개봉,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
5,6,마루 밑 아리에티,NaN,NaN,재개봉,2026-08-19,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
6,7,퍼피 구조대-더 다이노 무비,99%,2.9만,개봉,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
7,8,에이티즈 - 라이트 더 웨이 인 시네마,NaN,NaN,개봉예정,2026-08-19,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
8,9,호프,85%,446만,개봉,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
9,10,어떻게 해야 했을까,97%,6.5만,개봉,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...


# 전처리 결과 검증

In [10]:
quality = pd.DataFrame({
    "dtype": clean_df.dtypes.astype(str),
    "missing_count": clean_df.isnull().sum(),
    "missing_ratio(%)": (clean_df.isnull().mean() * 100).round(2),
})

display(quality)

print("중복 행:", clean_df.duplicated().sum())
print("영화명 결측:", clean_df["title"].isna().sum())

display(
    clean_df[
        ["rank", "egg_index", "cumulative_viewers_count"]
    ].head(10)
)


,dtype,missing_count,missing_ratio(%)
rank,Int64,0,0.00
title,string,0,0.00
egg_index,str,34,49.28
cumulative_viewers_count,str,30,43.48
is_re_release,str,0,0.00
release_date,str,39,56.52
age_rating,str,0,0.00
poster_url,str,0,0.00
crawled_at,str,0,0.00
source_url,str,0,0.00


중복 행: 0
영화명 결측: 0


,rank,egg_index,cumulative_viewers_count
0,1,98%,382.1만
1,2,97%,696.5만
2,3,97%,57.5만
3,4,93%,18.4만
4,5,77%,15.7만
5,6,NaN,NaN
6,7,99%,2.9만
7,8,NaN,NaN
8,9,85%,446만
9,10,97%,6.5만


# 최종 저장

- `data/raw/cgv_movie_raw.csv` : 중간 저장 원본
- `data/cgv_movie_clean.csv` : 최종 전처리 데이터

In [11]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"최종 CSV 저장 완료: {FINAL_CSV_PATH}")
print(f"최종 데이터 건수: {len(clean_df)}건")


최종 CSV 저장 완료: D:\AI\data_analytics\crawling\cgv_crawling\data\cgv_movie_clean.csv
최종 데이터 건수: 69건


#  MySQL 데이터베이스 저장

In [13]:
import os
from dotenv import load_dotenv
import pymysql
import pandas as pd

load_dotenv()

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_NAME = os.getenv("DB_NAME", "movie_chart")

# 1. DB 존재 확인 및 생성
server_conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    charset="utf8mb4",
    autocommit=True,
)

try:
    with server_conn.cursor() as cursor:
        cursor.execute(
            f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` "
            "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
        )
finally:
    server_conn.close()

# 2. DB 연결 및 데이터 저장
conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    charset="utf8mb4",
    autocommit=False,
)

try:
    with conn.cursor() as cursor:
        cursor.execute("DROP TABLE IF EXISTS cgv_movies")

        create_table_sql = """
        CREATE TABLE cgv_movies (
            id INT AUTO_INCREMENT PRIMARY KEY,
            `rank` INT NULL,
            title VARCHAR(255) NOT NULL,
            egg_index VARCHAR(10) NULL,
            `cumulative_viewers_count` VARCHAR(20) NULL,
            is_re_release VARCHAR(20) NULL,
            release_date DATE NULL,
            age_rating VARCHAR(100) NULL,
            poster_url TEXT NULL,
            crawled_at DATETIME NULL,
            source_url TEXT NULL
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        cursor.execute(create_table_sql)

        insert_sql = """
        INSERT INTO cgv_movies
        (`rank`, title, egg_index, `cumulative_viewers_count`,
         is_re_release, release_date, age_rating, poster_url, crawled_at, source_url)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        db_columns = [
            "rank",
            "title",
            "egg_index",
            "cumulative_viewers_count",
            "is_re_release",
            "release_date",
            "age_rating",
            "poster_url",
            "crawled_at",
            "source_url",
        ]

        rows = []
        for _, row in clean_df[db_columns].iterrows():
            values = []
            for col in db_columns:
                value = row[col]
                if pd.isna(value):
                    value = None
                values.append(value)
            rows.append(tuple(values))

        cursor.executemany(insert_sql, rows)

    conn.commit()
    print(f"MySQL 저장 완료: {len(rows)}건")

except Exception as e:
    conn.rollback()
    print(f"저장 실패: {e}")
    raise

finally:
    conn.close()

MySQL 저장 완료: 69건


# 최종 결과 확인

In [18]:
result_df = pd.read_csv(FINAL_CSV_PATH, encoding="utf-8-sig")

expected_columns = [
    "rank", "title", "egg_index", "cumulative_viewers_count", "is_re_release",
    "release_date", "age_rating", "poster_url", "crawled_at", "source_url"
]

print("최종 CSV 행/열:", result_df.shape)
print("최종 컬럼:", result_df.columns.tolist())
assert result_df.columns.tolist() == expected_columns
print("컬럼 검증: 정상 (10개)")
display(result_df.head(10))


최종 CSV 행/열: (69, 10)
최종 컬럼: ['rank', 'title', 'egg_index', 'cumulative_viewers_count', 'is_re_release', 'release_date', 'age_rating', 'poster_url', 'crawled_at', 'source_url']
컬럼 검증: 정상 (10개)


,rank,title,egg_index,cumulative_viewers_count,is_re_release,release_date,age_rating,poster_url,crawled_at,source_url
0,1,오디세이,98%,382.1만,개봉,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
1,2,스파이더맨-브랜드 뉴 데이,97%,696.5만,개봉,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
2,3,사랑의 하츄핑-고래보석의 전설,97%,57.5만,개봉,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
3,4,명탐정 코난-하이웨이의 타천사,93%,18.4만,개봉,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
4,5,오케이 마담2,77%,15.7만,개봉,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
5,6,마루 밑 아리에티,NaN,NaN,재개봉,2026-08-19,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
6,7,퍼피 구조대-더 다이노 무비,99%,2.9만,개봉,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
7,8,에이티즈 - 라이트 더 웨이 인 시네마,NaN,NaN,개봉예정,2026-08-19,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
8,9,호프,85%,446만,개봉,NaN,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
9,10,어떻게 해야 했을까,97%,6.5만,개봉,NaN,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...


In [15]:
conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    charset="utf8mb4",
    cursorclass=DictCursor
)

try:
    with conn.cursor() as cursor:
        cursor.execute("SELECT COUNT(*) AS cnt FROM cgv_movies")
        db_count = cursor.fetchone()["cnt"]

        cursor.execute(
            "SELECT `rank`, title, egg_index, `cumulative_viewers_count`, "
            "age_rating, poster_url, crawled_at, source_url "
            "FROM cgv_movies ORDER BY `rank` LIMIT 10"
        )
        db_preview = pd.DataFrame(cursor.fetchall())

    print(f"CSV 저장 건수: {len(clean_df)}건")
    print(f"MySQL 저장 건수: {db_count}건")
    print("저장 결과:", "정상" if db_count == len(clean_df) else "확인 필요")
    display(db_preview)
finally:
    conn.close()


CSV 저장 건수: 69건
MySQL 저장 건수: 69건
저장 결과: 정상


,rank,title,egg_index,cumulative_viewers_count,age_rating,poster_url,crawled_at,source_url
0,1,오디세이,98%,382.1만,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
1,2,스파이더맨-브랜드 뉴 데이,97%,696.5만,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
2,3,사랑의 하츄핑-고래보석의 전설,97%,57.5만,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
3,4,명탐정 코난-하이웨이의 타천사,93%,18.4만,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
4,5,오케이 마담2,77%,15.7만,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
5,6,마루 밑 아리에티,NaN,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
6,7,퍼피 구조대-더 다이노 무비,99%,2.9만,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
7,8,에이티즈 - 라이트 더 웨이 인 시네마,NaN,NaN,전체 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
8,9,호프,85%,446만,15세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...
9,10,어떻게 해야 했을까,97%,6.5만,12세 관람가,https://cdn.cgv.co.kr/cgvpomsfilm/Movie/Thumbn...,2026-08-16 16:08:07,https://cgv.co.kr/cnm/cgvChart/movieChart?tabP...


# 간단한 EDA 및 산출물 검증

In [17]:
if not clean_df.empty:
    display(
        clean_df[
            ["rank", "title", "egg_index", "cumulative_viewers_count", "age_rating"]
        ].head(10)
    )

,rank,title,egg_index,cumulative_viewers_count,age_rating
0,1,오디세이,98%,382.1만,15세 관람가
1,2,스파이더맨-브랜드 뉴 데이,97%,696.5만,12세 관람가
2,3,사랑의 하츄핑-고래보석의 전설,97%,57.5만,전체 관람가
3,4,명탐정 코난-하이웨이의 타천사,93%,18.4만,12세 관람가
4,5,오케이 마담2,77%,15.7만,15세 관람가
5,6,마루 밑 아리에티,NaN,NaN,전체 관람가
6,7,퍼피 구조대-더 다이노 무비,99%,2.9만,전체 관람가
7,8,에이티즈 - 라이트 더 웨이 인 시네마,NaN,NaN,전체 관람가
8,9,호프,85%,446만,15세 관람가
9,10,어떻게 해야 했을까,97%,6.5만,12세 관람가


# 최종 정리

## 구현 과정

**사이트 선택 → 동적 크롤링 → 추출·수집 → Raw CSV 중간 저장 → EDA → 전처리 → CSV 저장 → MySQL DB 저장 → 결과 확인**

## 산출물

1. **Notebook:** `.ipynb`
2. **Raw CSV:** `data/raw/cgv_movie_raw.csv`
3. **최종 CSV:** `data/cgv_movie_clean.csv`
4. **MySQL:** `movie_chart.cgv_movies`

## 최종 데이터 주요 컬럼

`rank, title, egg_index, cumulative_viewers_count, is_re_release, release_date, age_rating, poster_url, crawled_at, source_url`